In [ ]:
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import sys
from neuralop.models import FNO, TFNO
from neuralop import Trainer
from neuralop.training import AdamW
from neuralop.utils import count_model_params, get_project_root
from neuralop import LpLoss, H1Loss
from pathlib import Path

from burgers import load_burgers_1dtime

device = "cuda:0"

n_train = 40000
n_val = 500

save_path = Path("burgers_data") / f"train{n_train}_test{n_val}"

train_loader, val_loader, data_processor = load_burgers_1dtime(
    path=save_path,
    batch_size=64,
    test_batch_size=n_val,
)

val_loaders = {}
val_loaders[100] = val_loader

model = TFNO(
    n_modes=(16,),
    in_channels=1,
    out_channels=1,
    hidden_channels=64,
    n_layers=4,
)
model = model.to(device)

# Count and display the number of parameters
n_params = count_model_params(model)
print(f"\nOur model has {n_params} parameters.")
sys.stdout.flush()

optimizer = AdamW(
    model.parameters(), lr=1e-3, weight_decay=1e-4
)  # weight_decay can be tuned
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=1000, eta_min=1e-8
)

l2loss = LpLoss(d=1, p=2)  # L2 loss for function values
h1loss = H1Loss(d=1)  # H1 loss includes gradient information

train_loss = h1loss
eval_losses = {"h1": h1loss, "l2": l2loss}

print("\n### MODEL ###\n", model)
print("\n### OPTIMIZER ###\n", optimizer)
print("\n### SCHEDULER ###\n", scheduler)
print("\n### LOSSES ###")
print(f"\n * Train: {train_loss}")
print(f"\n * Test: {eval_losses}")
sys.stdout.flush()

trainer = Trainer(
    model=model,
    n_epochs=1000,
    device=device,
    data_processor=data_processor().to(device),
    wandb_log=True,  # Disable Weights & Biases logging for this tutorial
    eval_interval=20,  # Evaluate every 5 epochs
    use_distributed=False,  # Single GPU/CPU training
    verbose=True,  # Print training progress
)

trainer.train(
    train_loader=train_loader,
    test_loaders=val_loaders,
    optimizer=optimizer,
    scheduler=scheduler,
    regularizer=False,
    training_loss=train_loss,
    eval_losses=eval_losses,
)

In [ ]:
save_dir = Path("checkpoints")
save_dir.mkdir(exist_ok=True)

model_path = save_dir / "tfno_burgers_1d.pt"
torch.save(model.state_dict(), model_path)

print(f"Model saved to {model_path}")

In [ ]:
# Testing with randomly drawn parameters a

import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt


def solve_burgers_multiple(xs, a_coeffs, nu=0.05, nx=1000):
    x_vals = np.linspace(0, 1, nx)
    dx = x_vals[1] - x_vals[0]

    u0 = np.zeros(nx)
    for l, a_l in enumerate(a_coeffs, start=1):
        u0 += a_l * np.sin(l * np.pi * x_vals)
    u0[0] = u0[-1] = 0.0

    def burgers_rhs(t, u):
        dudt = np.zeros_like(u)
        dudt[1:-1] = (
            -u[1:-1] * (u[2:] - u[:-2]) / (2 * dx)
            + nu * (u[2:] - 2 * u[1:-1] + u[:-2]) / dx**2
        )
        return dudt

    sol = solve_ivp(
        burgers_rhs,
        (0.0, 1.0),
        u0,
        t_eval=[1.0],
        method="Radau",
        rtol=1e-8,
        atol=1e-10,
    )

    u_T = sol.y[:, 0]  # solution at t=1

    # interpolate for all xs
    return np.interp(xs, x_vals, u_T).reshape(-1, 1)


def compute_reference(a_coeffs, x_grid, device):
    xs = x_grid.cpu().numpy()
    a_np = a_coeffs.cpu().numpy()

    refs = [solve_burgers_multiple(xs, a_np[j]).flatten() for j in range(a_np.shape[0])]

    refs = np.stack(refs, axis=0)  # (N, G)
    return torch.from_numpy(refs).to(device).unsqueeze(1)  # (N, 1, G)


def build_initial_condition(a_coeffs, x_grid):
    N, n_modes = a_coeffs.shape
    G = x_grid.numel()

    u0 = torch.zeros(N, G, device=a_coeffs.device)

    for l in range(n_modes):
        u0 += a_coeffs[:, l].unsqueeze(1) * torch.sin((l + 1) * torch.pi * x_grid)

    u0[:, 0] = 0.0
    u0[:, -1] = 0.0

    return u0.unsqueeze(1)  # (N, 1, G)


N_eval = 10**1
n_modes = 9

a_coeffs = 2.0 * torch.rand(N_eval, n_modes) - 1.0

torch.set_grad_enabled(False)

G = 100
x_grid = torch.linspace(0, 1, G, device="cpu")

u_ref = compute_reference(a_coeffs, x_grid, device)

x_grid = x_grid.to(device)
a_coeffs = a_coeffs.to(device)
u0_eval = build_initial_condition(a_coeffs, x_grid)

model = TFNO(
    n_modes=(16,),
    in_channels=1,
    out_channels=1,
    hidden_channels=64,
    n_layers=4,
).to(device)

model.load_state_dict(torch.load("checkpoints/tfno_burgers_1d.pt"))
model.eval()

u_pred = model(u0_eval)

err = torch.norm(u_ref - u_pred, dim=(-1, -2)) / torch.norm(u_ref, dim=(-1, -2))

err_np = err.detach().cpu().numpy()

plt.figure(figsize=(8, 2.5))
plt.boxplot(
    err_np,
    vert=False,
    widths=0.5,
    patch_artist=True,
    boxprops=dict(facecolor="lightblue", color="blue"),
    medianprops=dict(color="darkblue"),
    whiskerprops=dict(color="blue"),
    capprops=dict(color="blue"),
    flierprops=dict(
        markerfacecolor="blue",
        marker="o",
        markersize=3,
        linestyle="none",
        markeredgecolor="none",
    ),
)
plt.xscale("log")
plt.xlabel("Relative $L^2$ error (log scale)", fontsize=12)
plt.title("Boxplot of relative $L^2$ Errors for the 1D Burgers equation", fontsize=13)
plt.grid(True, axis="x", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Use this if you have a precomputed set of parameters a for comparability

import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from neuralop.models import TFNO


def solve_burgers_multiple(xs, a_coeffs, nu=0.05, nx=1000):
    x_vals = np.linspace(0, 1, nx)
    dx = x_vals[1] - x_vals[0]

    u0 = np.zeros(nx)
    for l, a_l in enumerate(a_coeffs, start=1):
        u0 += a_l * np.sin(l * np.pi * x_vals)
    u0[0] = u0[-1] = 0.0

    def burgers_rhs(t, u):
        dudt = np.zeros_like(u)
        dudt[1:-1] = (
            -u[1:-1] * (u[2:] - u[:-2]) / (2 * dx)
            + nu * (u[2:] - 2 * u[1:-1] + u[:-2]) / dx**2
        )
        return dudt

    sol = solve_ivp(
        burgers_rhs,
        (0.0, 1.0),
        u0,
        t_eval=[1.0],
        method="Radau",
        rtol=1e-8,
        atol=1e-10,
    )

    u_T = sol.y[:, 0]
    return np.interp(xs, x_vals, u_T).reshape(-1, 1)


def compute_reference(a_coeffs, x_grid, device):
    xs = x_grid.cpu().numpy()
    a_np = a_coeffs.cpu().numpy()

    refs = [solve_burgers_multiple(xs, a_np[j]).flatten() for j in range(a_np.shape[0])]
    refs = np.stack(refs, axis=0)

    return torch.from_numpy(refs).to(device=device, dtype=torch.float32).unsqueeze(1)


def build_initial_condition(a_coeffs, x_grid):
    N, n_modes = a_coeffs.shape
    G = x_grid.numel()

    u0 = torch.zeros(N, G, device=a_coeffs.device, dtype=a_coeffs.dtype)

    for l in range(n_modes):
        u0 += a_coeffs[:, l].unsqueeze(1) * torch.sin((l + 1) * torch.pi * x_grid)

    u0[:, 0] = 0.0
    u0[:, -1] = 0.0

    return u0.unsqueeze(1)


device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_grad_enabled(False)

currentdir = os.getcwd()
parentdir = os.path.dirname(currentdir)

testset_path = os.path.join(parentdir, "heat_testset_10000.npy")
checkpoint_path = "checkpoints/tfno_burgers_1d.pt"

a_test = np.load(testset_path)
a_coeffs = torch.tensor(a_test, dtype=torch.float32)

G = 100
x_grid_cpu = torch.linspace(0, 1, G, device="cpu")

u_ref = compute_reference(a_coeffs, x_grid_cpu, device)

x_grid = x_grid_cpu.to(device)
a_coeffs = a_coeffs.to(device)
u0_eval = build_initial_condition(a_coeffs, x_grid)

model = TFNO(
    n_modes=(16,),
    in_channels=1,
    out_channels=1,
    hidden_channels=64,
    n_layers=4,
).to(device)

model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()

u_pred = model(u0_eval)

err = torch.norm(u_ref - u_pred, dim=(-1, -2))  # / torch.norm(u_ref, dim=(-1, -2))
err_np = err.detach().cpu().numpy()

plt.figure(figsize=(8, 2.5))
plt.boxplot(
    err_np,
    vert=False,
    widths=0.5,
    patch_artist=True,
    boxprops=dict(facecolor="lightblue", color="blue"),
    medianprops=dict(color="darkblue"),
    whiskerprops=dict(color="blue"),
    capprops=dict(color="blue"),
    flierprops=dict(
        markerfacecolor="blue",
        marker="o",
        markersize=3,
        linestyle="none",
        markeredgecolor="none",
    ),
)
plt.xscale("log")
plt.xlabel("Relative $L^2$ error (log scale)", fontsize=12)
plt.title("Boxplot of relative $L^2$ Errors for the 1D Burgers equation", fontsize=13)
plt.grid(True, axis="x", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()